In [18]:
import duckdb
conn = duckdb.connect()

In [19]:
conn.execute("""
CREATE OR REPLACE VIEW customers AS
SELECT *
FROM read_csv_auto('customers.csv');
""")

conn.execute("""
CREATE OR REPLACE VIEW sellers AS
SELECT *
FROM read_csv_auto('sellers.csv');
""")

conn.execute("""
CREATE OR REPLACE VIEW products AS
SELECT *
FROM read_csv_auto('products.csv');
""")

conn.execute("""
CREATE OR REPLACE VIEW orders AS
SELECT *
FROM read_csv_auto('orders.csv');
""")

conn.execute("""
CREATE OR REPLACE VIEW order_items AS
SELECT *
FROM read_csv_auto('order_items.csv');
""")

conn.execute("""
CREATE OR REPLACE VIEW shipments AS
SELECT *
FROM read_csv_auto('shipments.csv');
""")

In [20]:
# sql
b1 = conn.sql("""
WITH order_gmv AS (
    SELECT
        order_id,
        SUM(quantity * unit_price) AS gmv
    FROM order_items
    GROUP BY order_id
),

delivered_orders AS (
    SELECT
        o.order_id,
        o.customer_id,
        DATE_TRUNC('month', o.created_at) AS month,
        c.city,

        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.created_at
        ) AS delivered_order_num

    FROM orders o
    JOIN customers c
      ON o.customer_id = c.customer_id

    WHERE o.status = 'Delivered'
),

monthly_metrics AS (

    SELECT
        d.month,
        d.city,

        SUM(g.gmv) AS gmv,

        COUNT(DISTINCT d.order_id) AS number_of_orders,

        COUNT(DISTINCT d.customer_id) AS unique_customers,

        COUNT(
            DISTINCT CASE
                WHEN delivered_order_num >= 2
                THEN d.customer_id
            END
        ) AS repeat_customers

    FROM delivered_orders d
    JOIN order_gmv g
      ON d.order_id = g.order_id

    GROUP BY 1,2
)

SELECT
    month,
    city,
    gmv,
    number_of_orders,
    unique_customers,

    ROUND(
        repeat_customers * 1.0
        / NULLIF(unique_customers,0),
        4
    ) AS repeat_purchase_rate

FROM monthly_metrics

ORDER BY month, city;
""").df()

b1.head()

,month,city,gmv,number_of_orders,unique_customers,repeat_purchase_rate
0,2024-07-01,Ahmedabad,7450923.0,220,202,0.0842
1,2024-07-01,Bangalore,22641100.0,653,585,0.1060
2,2024-07-01,Chandigarh,5219993.0,130,111,0.1622
3,2024-07-01,Chennai,15670622.0,412,371,0.0943
4,2024-07-01,Delhi,24710629.0,741,651,0.1275


In [21]:
b2 = conn.sql("""
with monthly_segment_gmv as (
    SELECT
        DATE_TRUNC('month', o.created_at) AS month,
        c.segment,
        SUM(oi.quantity * oi.unit_price) AS gmv
    FROM orders o
    JOIN customers c
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON oi.order_id = o.order_id
GROUP BY 1,2
ORDER BY 1,2)

SELECT
    month,
    segment,
    gmv AS current_month_gmv,
    -- Grab the GMV from the previous month for the same segment
    LAG(gmv) OVER(PARTITION BY segment ORDER BY month) AS previous_month_gmv,
    
    -- Calculate the Absolute Growth 
    gmv - LAG(gmv) OVER(PARTITION BY segment ORDER BY month) AS absolute_growth,
    
    -- Calculate the Growth Rate Percentage (MoM %)
    ROUND(
        ((gmv - LAG(gmv) OVER(PARTITION BY segment ORDER BY month)) / 
        NULLIF(LAG(gmv) OVER(PARTITION BY segment ORDER BY month), 0)) * 100, 
        2
    ) AS mom_growth_percentage
FROM monthly_segment_gmv
ORDER BY month, segment;

""").df()
b2.head(10)

,month,segment,current_month_gmv,previous_month_gmv,absolute_growth,mom_growth_percentage
0,2024-07-01,Budget,52533037.0,NaN,NaN,NaN
1,2024-07-01,Premium,70991901.0,NaN,NaN,NaN
2,2024-07-01,Value,71036626.0,NaN,NaN,NaN
3,2024-08-01,Budget,48951681.0,52533037.0,-3581356.0,-6.82
4,2024-08-01,Premium,72719940.0,70991901.0,1728039.0,2.43
5,2024-08-01,Value,73160656.0,71036626.0,2124030.0,2.99
6,2024-09-01,Budget,49968923.0,48951681.0,1017242.0,2.08
7,2024-09-01,Premium,66338194.0,72719940.0,-6381746.0,-8.78
8,2024-09-01,Value,70057674.0,73160656.0,-3102982.0,-4.24
9,2024-10-01,Budget,50681358.0,49968923.0,712435.0,1.43


In [25]:
b3 = conn.sql("""
              WITH base AS (

    SELECT

        oi.seller_id,
        s.carrier,
        s.ship_to_city,

        o.order_id,

        (oi.quantity * oi.unit_price) AS item_gmv,

        CASE
            WHEN s.delivery_status <> 'OnTime'
            THEN 1
            ELSE 0
        END AS is_delayed,

        CASE
            WHEN s.delivery_status <> 'OnTime'
            THEN (
                s.delivered_at::date
                - o.promised_delivery_date::date
            )
        END AS delay_days

    FROM order_items oi

    JOIN orders o
        ON o.order_id = oi.order_id

    JOIN shipments s
        ON s.order_id = o.order_id

    WHERE o.status = 'Delivered'
)

SELECT

    seller_id,
    carrier,
    ship_to_city,

    SUM(item_gmv) AS total_gmv,

    SUM(
        CASE
            WHEN is_delayed = 1
            THEN item_gmv
            ELSE 0
        END
    ) AS delayed_gmv,

    ROUND(
        AVG(is_delayed::numeric),
        4
    ) AS delayed_order_rate,

    ROUND(
        AVG(delay_days),
        2
    ) AS avg_delay_days

FROM base

GROUP BY
    seller_id,
    carrier,
    ship_to_city

HAVING COUNT(DISTINCT order_id) >= 15

ORDER BY delayed_order_rate DESC;
""").df()
b3.head(10)

,seller_id,carrier,ship_to_city,total_gmv,delayed_gmv,delayed_order_rate,avg_delay_days
0,S0050,Ekart,Kolkata,117954.0,106937.0,0.8235,1.79
1,S0012,Delhivery,Chennai,495670.0,484850.0,0.8125,0.69
2,S0009,BlueDart,Chennai,243920.0,232624.0,0.8000,0.42
3,S0166,Ekart,Kolkata,178133.0,163313.0,0.8000,1.50
4,S0009,Ekart,Delhi,166506.0,142585.0,0.7778,0.21
5,S0005,BlueDart,Mumbai,151587.0,140461.0,0.7500,0.58
6,S0013,Ekart,Pune,238242.0,186915.0,0.7500,0.33
7,S0013,Delhivery,Bangalore,282747.0,168419.0,0.7368,0.57
8,S0015,Ekart,Chennai,392225.0,202985.0,0.7059,0.50
9,S0004,Delhivery,Delhi,596892.0,388848.0,0.7059,0.58


In [24]:
b4 = conn.sql("""

with first_delivered_order as(
             select * from (
select o."customer_id", o."order_id", o."created_at",
case when s."delivery_status" = 'OnTime' 
             then 'OnTime'
             else 'Delayed'
             end as "first_order_delay_status",
             row_number() over(partition by o."customer_id" order by o."created_at") as rn
             from orders o
             join shipments s
             on o."order_id" = s."order_id"
             where o."status" = 'Delivered' ) x
             where rn = 1),

repeat_within_90_days as(
             select f."customer_id", f."first_order_delay_status", 
             case 
             when exists(
             select 1 from orders o2
             where o2."customer_id" = f."customer_id"
             and o2."status" = 'Delivered'
             and o2."created_at">f."created_at"
             and o2."created_at" <= f."created_at" + interval '90 days')           
              then 1
                else 0
             end as repeat_within_90d
             from first_delivered_order f)
select "first_order_delay_status",
             round(avg(repeat_within_90d::numeric),4) as repeat_rate_90d
             from repeat_within_90_days
             group by "first_order_delay_status"
             
             
             
             
             
             """).df()
b4.head()

,first_order_delay_status,repeat_rate_90d
0,Delayed,0.3934
1,OnTime,0.4048
